# Preprocessing and Multi-Modal Feature Engineering
This notebook transforms raw text data into structured features using four primary approaches:
1. **Text Normalization:** Context-aware cleaning preserving negations and rules.
2. **Sparse Vectorization:** Traditional Bag-of-Words (BoW) and TF-IDF representations.
3. **Semantic Embeddings:** Deep learning-based document embeddings using DistilBERT.
4. **Contextual Similarity:** Measuring the semantic distance between content and rules via Cosine Similarity.

## Data Acquisition
The dataset is retrieved directly from the Kaggle competition using the Kaggle API.

To reproduce this environment:
1. Upload your `kaggle.json` API token.
2. Run the following commands to download and extract the data:

```bash
# !pip install kaggle
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c jigsaw-agile-community-rules
# !unzip jigsaw-agile-community-rules.zip -d data

In [25]:
import gc
import os
import pandas as pd
import numpy as np

from scipy.sparse import hstack, csr_matrix

import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

### 1. Context-Preserving Text Cleaning
Unlike standard NLP pipelines, we preserve negations (no, not, never) because they are critical for detecting community rule violations.

In [27]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
# Critical negations to keep for rule context
exclude_from_stop = {"not", "no", "never", "none", "neither", "nor", "against", "violation", "violate"}
stop_words = stop_words - exclude_from_stop

def clean_stop_words(text):
    if pd.isna(text) or text == "": return ""
    return " ".join([word for word in str(text).split() if word.lower() not in stop_words])

# Apply to all text columns
text_columns = ['body', 'rule', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2']
for col in text_columns:
    train[col] = train[col].fillna("").apply(clean_stop_words)
    test[col] = test[col].fillna("").apply(clean_stop_words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### 2. Sparse Vectorization (BoW & TF-IDF)
Combining frequency-based (CountVectorizer) and importance-based (TfidfVectorizer) representations to capture lexical patterns.

In [28]:
# TRAIN

train_vocabulary_source = pd.concat([
    train["body"],
    train["positive_example_1"],
    train["positive_example_2"],
    train["negative_example_1"],
    train["negative_example_2"]
])

# BoW

bow_vectorizer = CountVectorizer(token_pattern=r'(?u)\b[a-zA-Z]{2,}\b', stop_words='english', min_df=2)
bow_vectorizer.fit(train_vocabulary_source)

X_train_body_bow = bow_vectorizer.transform(train["body"])
X_train_pos1_bow = bow_vectorizer.transform(train["positive_example_1"])
X_train_pos2_bow = bow_vectorizer.transform(train["positive_example_2"])
X_train_neg1_bow = bow_vectorizer.transform(train["negative_example_1"])
X_train_neg2_bow = bow_vectorizer.transform(train["negative_example_2"])


# TF IDF

tfidf_vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b[a-zA-Z]{2,}\b', stop_words='english', min_df=2)
tfidf_vectorizer.fit(train_vocabulary_source)

X_train_body_tfidf = tfidf_vectorizer.transform(train["body"])
X_train_pos1_tfidf = tfidf_vectorizer.transform(train["positive_example_1"])
X_train_pos2_tfidf = tfidf_vectorizer.transform(train["positive_example_2"])
X_train_neg1_tfidf = tfidf_vectorizer.transform(train["negative_example_1"])
X_train_neg2_tfidf = tfidf_vectorizer.transform(train["negative_example_2"])


X_train_bow_tfidf = hstack([
    hstack([X_train_body_bow, X_train_body_tfidf]),
    hstack([X_train_pos1_bow, X_train_pos1_tfidf]),
    hstack([X_train_pos2_bow, X_train_pos2_tfidf]),
    hstack([X_train_neg1_bow, X_train_neg1_tfidf]),
    hstack([X_train_neg2_bow, X_train_neg2_tfidf])
])


# TEST

# BoW

X_test_body_bow = bow_vectorizer.transform(test["body"])
X_test_pos1_bow = bow_vectorizer.transform(test["positive_example_1"])
X_test_pos2_bow = bow_vectorizer.transform(test["positive_example_2"])
X_test_neg1_bow = bow_vectorizer.transform(test["negative_example_1"])
X_test_neg2_bow = bow_vectorizer.transform(test["negative_example_2"])


# TF-IDF

X_test_body_tfidf = tfidf_vectorizer.transform(test["body"])
X_test_pos1_tfidf = tfidf_vectorizer.transform(test["positive_example_1"])
X_test_pos2_tfidf = tfidf_vectorizer.transform(test["positive_example_2"])
X_test_neg1_tfidf = tfidf_vectorizer.transform(test["negative_example_1"])
X_test_neg2_tfidf = tfidf_vectorizer.transform(test["negative_example_2"])


X_test_bow_tfidf = hstack([
    hstack([X_test_body_bow, X_test_body_tfidf]),
    hstack([X_test_pos1_bow, X_test_pos1_tfidf]),
    hstack([X_test_pos2_bow, X_test_pos2_tfidf]),
    hstack([X_test_neg1_bow, X_test_neg1_tfidf]),
    hstack([X_test_neg2_bow, X_test_neg2_tfidf])
])

### 3. Deep Semantic Embeddings
Using a fine-tuned DistilBERT model to extract the [CLS] token embeddings. This captures the deep semantic meaning of the content and the rules.

In [29]:
MODEL_PATH = "BernaTS/distilbert-5channel-jigsaw"

# Initializing Tokenizer and Model
# local_files_only=True ensures no external requests are made during inference
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModel.from_pretrained(MODEL_PATH).to(device)
model.eval()

def get_doc_embedding(texts, tokenizer, model, device, batch_size=32):
    """
    Generates high-quality document embeddings using the [CLS] token of a pre-trained Transformer.

    This function is optimized for large datasets with explicit memory management
    to prevent OOM (Out of Memory) issues in resource-constrained environments.

    Args:
        texts (list): A list of text strings to be embedded.
        tokenizer: The HuggingFace tokenizer instance.
        model: The HuggingFace model instance (DistilBERT).
        device: The computing device (CPU or CUDA).
        batch_size (int): Number of samples per batch.

    Returns:
        np.ndarray: A matrix of shape (n_samples, hidden_dim) representing document embeddings.
    """
    model.eval()
    embeddings = []

    # Using tqdm for progress tracking during long extraction processes
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting Embeddings"):
        batch = texts[i : i + batch_size]

        # Tokenization with truncation and padding to ensure consistent input shapes
        inputs = tokenizer(
            list(batch),
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            # Extracting the [CLS] token (index 0) as the aggregate representation of the sentence
            cls_embeds = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()
            embeddings.append(cls_embeds)

        # Manual memory cleanup to ensure GPU stability
        del inputs, outputs
        if i % (batch_size * 5) == 0:
            torch.cuda.empty_cache()
            import gc; gc.collect()

    return np.vstack(embeddings)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [30]:
# TRAIN Embedding

X_train_rule_embed = get_doc_embedding(train["rule"].tolist(), tokenizer, model, device)
X_train_body_embed = get_doc_embedding(train["body"].tolist(), tokenizer, model, device)
X_train_pos1_embed = get_doc_embedding(train["positive_example_1"].tolist(), tokenizer, model, device)
X_train_pos2_embed = get_doc_embedding(train["positive_example_2"].tolist(), tokenizer, model, device)
X_train_neg1_embed = get_doc_embedding(train["negative_example_1"].tolist(), tokenizer, model, device)
X_train_neg2_embed = get_doc_embedding(train["negative_example_2"].tolist(), tokenizer, model, device)

X_train_rule_sparse = csr_matrix(X_train_rule_embed)
X_train_body_sparse = csr_matrix(X_train_body_embed)
X_train_pos1_sparse = csr_matrix(X_train_pos1_embed)
X_train_pos2_sparse = csr_matrix(X_train_pos2_embed)
X_train_neg1_sparse = csr_matrix(X_train_neg1_embed)
X_train_neg2_sparse = csr_matrix(X_train_neg2_embed)

embedding_block_train = hstack([
    X_train_rule_sparse,
    X_train_body_sparse,
    X_train_pos1_sparse,
    X_train_pos2_sparse,
    X_train_neg1_sparse,
    X_train_neg2_sparse
])


# TEST Embedding

X_test_rule_embed = get_doc_embedding(test["rule"].tolist(), tokenizer, model, device)
X_test_body_embed = get_doc_embedding(test["body"].tolist(), tokenizer, model, device)
X_test_pos1_embed = get_doc_embedding(test["positive_example_1"].tolist(), tokenizer, model, device)
X_test_pos2_embed = get_doc_embedding(test["positive_example_2"].tolist(), tokenizer, model, device)
X_test_neg1_embed = get_doc_embedding(test["negative_example_1"].tolist(), tokenizer, model, device)
X_test_neg2_embed = get_doc_embedding(test["negative_example_2"].tolist(), tokenizer, model, device)


X_test_rule_sparse = csr_matrix(X_test_rule_embed)
X_test_body_sparse = csr_matrix(X_test_body_embed)
X_test_pos1_sparse = csr_matrix(X_test_pos1_embed)
X_test_pos2_sparse = csr_matrix(X_test_pos2_embed)
X_test_neg1_sparse = csr_matrix(X_test_neg1_embed)
X_test_neg2_sparse = csr_matrix(X_test_neg2_embed)

embedding_block_test = hstack([
    X_test_rule_sparse,
    X_test_body_sparse,
    X_test_pos1_sparse,
    X_test_pos2_sparse,
    X_test_neg1_sparse,
    X_test_neg2_sparse
])

Extracting Embeddings:   0%|          | 0/64 [00:00<?, ?it/s]

Extracting Embeddings:   0%|          | 0/64 [00:00<?, ?it/s]

Extracting Embeddings:   0%|          | 0/64 [00:00<?, ?it/s]

Extracting Embeddings:   0%|          | 0/64 [00:00<?, ?it/s]

Extracting Embeddings:   0%|          | 0/64 [00:00<?, ?it/s]

...

Extracting Embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting Embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting Embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting Embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting Embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

### 4. Semantic Similarity Engineering (Cosine Similarity)
We calculate the cosine similarity between the 'Body' and the 'Rule' (and examples) to measure how closely the content relates to the specific community guidelines.

In [31]:
def safe_cosine(a, b):
    """
    Computes point-wise cosine similarity between two matrices for context matching.
    This optimized version performs row-by-row similarity instead of a full
    cross-product matrix to maintain a low memory footprint.

    Args:
        a (np.ndarray): First embedding matrix.
        b (np.ndarray): Second embedding matrix (must have same shape as a).

    Returns:
        np.ndarray: 1D array of similarity scores ranging from -1 to 1.
    """
    dot = np.sum(a * b, axis=1)
    norm_a = np.linalg.norm(a, axis=1)
    norm_b = np.linalg.norm(b, axis=1)
    # 1e-9 added to prevent division by zero for empty/padding strings
    return dot / (norm_a * norm_b + 1e-9)

In [32]:
# TRAIN Cosine Similarity
X_train_cos_sim_body_rule = safe_cosine(X_train_body_embed, X_train_rule_embed)
X_train_cos_sim_body_pos1 = safe_cosine(X_train_body_embed, X_train_pos1_embed)
X_train_cos_sim_rule_pos1 = safe_cosine(X_train_rule_embed, X_train_pos1_embed)
X_train_cos_sim_body_pos2 = safe_cosine(X_train_body_embed, X_train_pos2_embed)
X_train_cos_sim_rule_pos2 = safe_cosine(X_train_rule_embed, X_train_pos2_embed)
X_train_cos_sim_body_neg1 = safe_cosine(X_train_body_embed, X_train_neg1_embed)
X_train_cos_sim_rule_neg1 = safe_cosine(X_train_rule_embed, X_train_neg1_embed)
X_train_cos_sim_body_neg2 = safe_cosine(X_train_body_embed, X_train_neg2_embed)
X_train_cos_sim_rule_neg2 = safe_cosine(X_train_rule_embed, X_train_neg2_embed)

X_train_cos_sim = hstack([
    csr_matrix(X_train_cos_sim_body_rule).T,
    csr_matrix(X_train_cos_sim_body_pos1).T,
    csr_matrix(X_train_cos_sim_rule_pos1).T,
    csr_matrix(X_train_cos_sim_body_pos2).T,
    csr_matrix(X_train_cos_sim_rule_pos2).T,
    csr_matrix(X_train_cos_sim_body_neg1).T,
    csr_matrix(X_train_cos_sim_rule_neg1).T,
    csr_matrix(X_train_cos_sim_body_neg2).T,
    csr_matrix(X_train_cos_sim_rule_neg2).T,
])

# TEST Cosine Similarity
X_test_cos_sim_body_rule = safe_cosine(X_test_body_embed, X_test_rule_embed)
X_test_cos_sim_body_pos1 = safe_cosine(X_test_body_embed, X_test_pos1_embed)
X_test_cos_sim_rule_pos1 = safe_cosine(X_test_rule_embed, X_test_pos1_embed)
X_test_cos_sim_body_pos2 = safe_cosine(X_test_body_embed, X_test_pos2_embed)
X_test_cos_sim_rule_pos2 = safe_cosine(X_test_rule_embed, X_test_pos2_embed)
X_test_cos_sim_body_neg1 = safe_cosine(X_test_body_embed, X_test_neg1_embed)
X_test_cos_sim_rule_neg1 = safe_cosine(X_test_rule_embed, X_test_neg1_embed)
X_test_cos_sim_body_neg2 = safe_cosine(X_test_body_embed, X_test_neg2_embed)
X_test_cos_sim_rule_neg2 = safe_cosine(X_test_rule_embed, X_test_neg2_embed)

X_test_cos_sim = hstack([
    csr_matrix(X_test_cos_sim_body_rule).T,
    csr_matrix(X_test_cos_sim_body_pos1).T,
    csr_matrix(X_test_cos_sim_rule_pos1).T,
    csr_matrix(X_test_cos_sim_body_pos2).T,
    csr_matrix(X_test_cos_sim_rule_pos2).T,
    csr_matrix(X_test_cos_sim_body_neg1).T,
    csr_matrix(X_test_cos_sim_rule_neg1).T,
    csr_matrix(X_test_cos_sim_body_neg2).T,
    csr_matrix(X_test_cos_sim_rule_neg2).T
])

# Final Feature Integration
Heterogeneous feature channels (lexical, semantic, and similarity) are fused into a unified hybrid representation.

The final matrices are converted to CSR (Compressed Sparse Row) format to maximize memory efficiency and accelerate the training process.

In [33]:
X_train = hstack([X_train_bow_tfidf, embedding_block_train, X_train_cos_sim])
y_train = train["rule_violation"]

X_test = hstack([X_test_bow_tfidf, embedding_block_test, X_test_cos_sim])

X_train_csr = X_train.tocsr()
X_test_csr = X_test.tocsr()

# Data Serialization and Export
The final engineered artifacts are serialized to disk to ensure modularity.

By exporting these matrices, the model optimization phase can be conducted independently without re-executing the entire preprocessing pipeline.

In [34]:
import scipy.sparse as sp
import joblib
import os

groups = train["subreddit"].values

def save_processed_artifacts(X_train, X_test, y, test_ids, groups, output_dir="data/processed"):
    """
    Serializes the processed datasets into compressed formats.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Sparse matrices
    sp.save_npz(os.path.join(output_dir, 'X_train_csr.npz'), X_train)
    sp.save_npz(os.path.join(output_dir, 'X_test_csr.npz'), X_test)

    # Metadata & Labels
    joblib.dump(y, os.path.join(output_dir, 'y_train.pkl'))
    joblib.dump(test_ids, os.path.join(output_dir, 'test_ids.pkl'))
    joblib.dump(groups, os.path.join(output_dir, 'groups.pkl'))

# Execution
save_processed_artifacts(X_train_csr, X_test_csr, y_train, test['row_id'], groups)

## Local Download (Optional)

Use the following block if you are working on Google Colab and want to download the processed artifacts to your local machine.

```bash
# from google.colab import files
# files.download('data/processed/X_train_csr.npz')
# files.download('data/processed/X_test_csr.npz')
# files.download('data/processed/y_train.pkl')
# files.download('data/processed/test_ids.pkl')
# files.download('data/processed/groups.pkl')